# Chapter 3.3 EB Baselines, Ablations, and Claim Audit

This split starts from the EB main model cache written by Chapter 3.2. It does not repeat the main training experiment; it uses that trained model to study solver behavior, CNF endpoint training cost, ablations, trajectory diagnostics, and final claim boundaries.

## Tutorial setup

The setup cells keep only reusable infrastructure here: project-root discovery, deterministic seeds, output directories, and save/display helpers. The scientific objects are introduced in the experiment sections below.

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

try:
    import torch
except ImportError as exc:
    raise ImportError("This notebook requires PyTorch for CFM training and sampling.") from exc

CWD = Path.cwd().resolve()
root_candidates = [
    CWD,
    CWD.parent,
    CWD.parent.parent,
    CWD / "flow_matching_for_dynamic_biology",
    CWD.parent / "flow_matching_for_dynamic_biology",
    CWD.parent.parent / "flow_matching_for_dynamic_biology",
]
for candidate in root_candidates:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError("Could not locate project root containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ch03_tutorial as ch03
from src.utils import set_seed
from src.data import load_eb_timecourse_for_ch03, copy_trajectorynet_eb_to_data
from src.models import VelocityMLP, count_parameters
from src.losses import cfm_batch, cfm_loss_from_pairs
from src.train import train_cfm_steps
from src.sampling import euler_sample, midpoint_sample, odeint_sample
from src.metrics import endpoint_metrics, mmd_rbf, sliced_wasserstein_distance, trajectory_straightness
from src.flow_matching_reporting import plot_nfe_vs_error

In [ ]:
SEED = int(os.environ.get("CH03_SEED", "42"))
set_seed(SEED)
rng = np.random.default_rng(SEED)

QUICK_MODE = os.environ.get("CH03_QUICK", "1") == "1"
SMOKE_MODE = os.environ.get("CH03_SMOKE_MODE", "0") == "1"
PAPER_FIGURE_MODE = os.environ.get("CH03_PAPER_FIGURE_MODE", "1") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FULL_RUN_HINTS = {
    "eb_train_steps": 10000,
    "cnf_steps": 600,
    "time_ablation_steps": 1200,
    "capacity_steps": 1200,
    "notes": "Set CH03_QUICK=0 for full-run settings; default remains quick and reproducible.",
}

context = ch03.make_ch03_context(PROJECT_ROOT)
FIG_DIR = context.fig_dir
TABLE_DIR = context.table_dir
OUT_DIR = context.output_dir
PAPER_COLORS = ch03.PAPER_COLORS

figures_written = []
paper_ready_png_written = []
paper_ready_pdf_written = []
tables_written = []
paper_tables_written = []
skipped_items = []

set_paper_style = ch03.set_paper_style
add_panel_label = ch03.add_panel_label
short_strategy_label = ch03.short_strategy_label
clean_spines = ch03.clean_spines
format_metric_axis = ch03.format_metric_axis
add_note = ch03.add_note
as_np = ch03.as_np
subsample_idx = ch03.subsample_idx
robust_limits = ch03.robust_limits
format_axis = ch03.format_axis

set_paper_style()
print({"project_root": str(PROJECT_ROOT), "device": DEVICE, "quick_mode": QUICK_MODE, "smoke_mode": SMOKE_MODE, "seed": SEED})

In [ ]:
def _rel(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def save_figure(fig, filename, dpi=300, write_pdf=None):
    if write_pdf is None:
        write_pdf = PAPER_FIGURE_MODE
    png_path = ch03.save_figure(fig, FIG_DIR, filename, dpi=dpi, write_pdf=write_pdf)
    rel_png = _rel(png_path)
    if rel_png not in figures_written:
        figures_written.append(rel_png)
    if write_pdf:
        pdf_path = png_path.with_suffix(".pdf")
        rel_pdf = _rel(pdf_path)
        if rel_pdf not in paper_ready_pdf_written:
            paper_ready_pdf_written.append(rel_pdf)
    plt.close(fig)
    return png_path


def save_paper_figure(fig, stem, dpi=300):
    png_path = ch03.save_figure(fig, FIG_DIR, stem, dpi=dpi, write_pdf=True)
    pdf_path = png_path.with_suffix(".pdf")
    for path, bucket in [(png_path, paper_ready_png_written), (pdf_path, paper_ready_pdf_written)]:
        rel = _rel(path)
        if rel not in bucket:
            bucket.append(rel)
    rel_png = _rel(png_path)
    if rel_png not in figures_written:
        figures_written.append(rel_png)
    plt.close(fig)
    return png_path, pdf_path


def save_csv(frame, filename):
    path = ch03.save_csv(TABLE_DIR / filename, pd.DataFrame(frame))
    rel = _rel(path)
    if rel not in tables_written:
        tables_written.append(rel)
    return path


def save_paper_table(frame, stem, index=False):
    paths = ch03.save_paper_table(TABLE_DIR / stem, pd.DataFrame(frame), index=index)
    for path in paths:
        rel = _rel(path)
        if rel not in paper_tables_written:
            paper_tables_written.append(rel)
    return paths


def save_run_json(payload, filename):
    return ch03.save_json(OUT_DIR / filename, payload)


def display_saved_figure(path, width=None):
    return ch03.display_saved_figure(path, width=width)


def display_saved_figures(paths, width=None):
    return ch03.display_saved_figures(paths, width=width)


def display_table(frame, columns=None, n=10):
    return ch03.display_table(pd.DataFrame(frame), columns=columns, n=n)

## 1. Load EB inputs and the main-model cache

The baseline and ablation cells need the same EB split and PHATE projector as the main notebook. They load the saved main model instead of re-running the main training loop. If the cache is missing, run `03_2_eb20d_main_flow_matching.ipynb` first.

In [ ]:
EB_PATH = PROJECT_ROOT / "data" / "trajectorynet_eb" / "eb_velocity_v5.npz"
EB_SOURCE_PATH = PROJECT_ROOT.parent / "baselines" / "trajectorynet" / "data" / "eb_velocity_v5.npz"
if not EB_PATH.exists() and EB_SOURCE_PATH.exists():
    copy_trajectorynet_eb_to_data(EB_SOURCE_PATH, EB_PATH)

eb = load_eb_timecourse_for_ch03(EB_PATH, max_cells_per_time=900 if QUICK_MODE else None, seed=SEED)
cost_embedding = "X_cost"
plot_embedding = "X_plot"
n_cost_dims = int(eb[cost_embedding].shape[1])

time_counts = pd.Series(eb["time"].astype(str)).value_counts().sort_index()
time_order = list(time_counts.index)
source_time = time_order[0]
target_time = time_order[1]
mask_source = eb["time"].astype(str) == source_time
mask_target = eb["time"].astype(str) == target_time
X_source_20d = np.asarray(eb["X_cost"][mask_source], dtype=np.float32)
X_target_20d = np.asarray(eb["X_cost"][mask_target], dtype=np.float32)
X_source_phate = np.asarray(eb["X_plot"][mask_source], dtype=np.float32)
X_target_phate = np.asarray(eb["X_plot"][mask_target], dtype=np.float32)
count_preview = pd.DataFrame({"time": time_counts.index, "n_cells": time_counts.values})
display_table(count_preview, n=8)

In [ ]:
def train_val_indices(n, train_fraction=0.8, seed=42):
    local_rng = np.random.default_rng(seed)
    perm = local_rng.permutation(n)
    n_train = int(round(float(train_fraction) * n))
    n_train = min(max(n_train, 1), n - 1)
    train = np.sort(perm[:n_train])
    val = np.sort(perm[n_train:])
    return train, val

src_train_idx, src_val_idx = train_val_indices(len(X_source_20d), seed=SEED + 11)
tgt_train_idx, tgt_val_idx = train_val_indices(len(X_target_20d), seed=SEED + 12)
X0_train, X0_val = X_source_20d[src_train_idx], X_source_20d[src_val_idx]
X1_train, X1_val = X_target_20d[tgt_train_idx], X_target_20d[tgt_val_idx]
X0_train_phate, X0_val_phate = X_source_phate[src_train_idx], X_source_phate[src_val_idx]
X1_train_phate, X1_val_phate = X_target_phate[tgt_train_idx], X_target_phate[tgt_val_idx]

pair_rng = np.random.default_rng(SEED + 13)
def pair_batch_fn(batch_size):
    idx0 = pair_rng.integers(0, len(X0_train), size=int(batch_size))
    idx1 = pair_rng.integers(0, len(X1_train), size=int(batch_size))
    return {"x0": X0_train[idx0].astype(np.float32), "x1": X1_train[idx1].astype(np.float32)}

val_pair_n = 1200 if QUICK_MODE else 2500
if SMOKE_MODE:
    val_pair_n = 180
val_pair_rng = np.random.default_rng(SEED + 13)
val_i0 = val_pair_rng.integers(0, len(X0_val), size=int(val_pair_n))
val_i1 = val_pair_rng.integers(0, len(X1_val), size=int(val_pair_n))
val_x0 = X0_val[val_i0].astype(np.float32)
val_x1 = X1_val[val_i1].astype(np.float32)
val_t_grid = np.asarray([0.25, 0.50, 0.75], dtype=np.float32)
print({"X0_train": X0_train.shape, "X1_train": X1_train.shape, "X0_val": X0_val.shape, "X1_val": X1_val.shape})

In [ ]:
def val_cfm_mse(model, x0_np, x1_np, t_values, device="cpu", max_eval_pairs=None):
    model.eval()
    if max_eval_pairs is not None and len(x0_np) > max_eval_pairs:
        idx = subsample_idx(len(x0_np), max_eval_pairs, seed=SEED + 14)
        x0_np = x0_np[idx]
        x1_np = x1_np[idx]
    x0_t = torch.as_tensor(x0_np, dtype=torch.float32, device=device)
    x1_t = torch.as_tensor(x1_np, dtype=torch.float32, device=device)
    losses = []
    with torch.no_grad():
        for t_value in t_values:
            t_t = torch.full((x0_t.shape[0], 1), float(t_value), dtype=torch.float32, device=device)
            loss = cfm_loss_from_pairs(model, x0_t, x1_t, s=t_t)
            losses.append(float(loss.detach().cpu()))
    model.train()
    return float(np.mean(losses))

In [ ]:
MAIN_MODEL_PATH = OUT_DIR / f"ch03_eb20d_velocity_mlp_seed{SEED}.pt"
MAIN_CONFIG_PATH = OUT_DIR / f"ch03_eb20d_main_config_seed{SEED}.json"
if not MAIN_MODEL_PATH.exists() or not MAIN_CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Missing EB main model cache. Run notebooks/03_2_eb20d_main_flow_matching.ipynb before this baseline notebook."
    )
main_model_config = json.loads(MAIN_CONFIG_PATH.read_text())
eb_model = VelocityMLP(
    x_dim=int(main_model_config.get("x_dim", 20)),
    hidden_dim=int(main_model_config.get("hidden_dim", 256)),
    hidden_layers=int(main_model_config.get("hidden_layers", 4)),
).to(DEVICE)
eb_model.load_state_dict(torch.load(MAIN_MODEL_PATH, map_location=DEVICE))
eb_model.eval()
print({"loaded_model_cache": str(MAIN_MODEL_PATH.relative_to(PROJECT_ROOT)), "main_steps": main_model_config.get("steps")})

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_neighbors = min(20, len(eb["X_cost"]))
phate_projector = KNeighborsRegressor(n_neighbors=knn_neighbors, weights="distance")
phate_projector.fit(np.asarray(eb["X_cost"], dtype=np.float32), np.asarray(eb["X_plot"], dtype=np.float32))

sample_n = min(1200 if QUICK_MODE else 2200, len(X0_val))
if SMOKE_MODE:
    sample_n = min(150, len(X0_val))
sample_idx = subsample_idx(len(X0_val), sample_n, seed=SEED + 31)
x0_eval_20d = torch.as_tensor(X0_val[sample_idx], dtype=torch.float32, device=DEVICE)
target_eval_20d = X1_val.astype(np.float32)
print({"eval_source_cells": int(sample_n), "target_eval_cells": int(len(target_eval_20d)), "projection": "kNN X_cost to X_plot for display only"})

## 8. Solver Comparison Lite

This first version compares fixed-step Euler, fixed-step midpoint, and `dopri5` when `torchdiffeq` is available. The comparison reports NFE, wall-clock time, MMD in 20D, and Sliced W2 in 20D.

This is not a CNF training baseline. It is only a post-training sampling diagnostic for the same trained 20D CFM vector field.

Solver comparison asks how numerical integration choices affect endpoint metrics after the model has already been trained.

In [ ]:
solver_rows = []
solver_specs = [("euler", K) for K in [5, 10, 20, 50, 100]] + [("midpoint", K) for K in [5, 10, 20, 50]]
if SMOKE_MODE:
    solver_specs = [("euler", K) for K in [5, 10]] + [("midpoint", K) for K in [5, 10]]

for solver_name, K in solver_specs:
    tic = time.perf_counter()
    if solver_name == "euler":
        endpoint_t, traj_t, nfe = euler_sample(eb_model, x0_eval_20d, n_steps=K, return_traj=True)
    elif solver_name == "midpoint":
        endpoint_t, traj_t, nfe = midpoint_sample(eb_model, x0_eval_20d, n_steps=K, return_traj=True)
    else:
        raise ValueError(solver_name)
    wall = time.perf_counter() - tic
    endpoint_20d = as_np(endpoint_t)
    traj_20d = as_np(traj_t)
    solver_rows.append({
        "sampler": solver_name,
        "steps": int(K),
        "rtol": np.nan,
        "nfe": int(nfe),
        "wall_time_sec": float(wall),
        "mmd_20d": float(mmd_rbf(endpoint_20d, target_eval_20d)),
        "sliced_w2_20d": float(sliced_wasserstein_distance(endpoint_20d, target_eval_20d, n_projections=128 if not SMOKE_MODE else 48, seed=SEED + K + (0 if solver_name == "euler" else 1000))),
        "trajectory_straightness_20d": float(trajectory_straightness(traj_20d)),
    })

rtols = [1e-3, 1e-5]
if SMOKE_MODE:
    rtols = [1e-3]
for rtol in rtols:
    try:
        tic = time.perf_counter()
        endpoint_t, traj_t, nfe = odeint_sample(eb_model, x0_eval_20d, rtol=rtol, atol=rtol, method="dopri5")
        wall = time.perf_counter() - tic
        endpoint_20d = as_np(endpoint_t)
        traj_20d = as_np(traj_t)
        solver_rows.append({
            "sampler": "dopri5",
            "steps": "adaptive",
            "rtol": float(rtol),
            "nfe": int(nfe),
            "wall_time_sec": float(wall),
            "mmd_20d": float(mmd_rbf(endpoint_20d, target_eval_20d)),
            "sliced_w2_20d": float(sliced_wasserstein_distance(endpoint_20d, target_eval_20d, n_projections=128 if not SMOKE_MODE else 48, seed=SEED + int(-np.log10(rtol)))),
            "trajectory_straightness_20d": float(trajectory_straightness(traj_20d)),
        })
    except ImportError as exc:
        message = f"torchdiffeq unavailable; skipped dopri5 rtol={rtol}: {exc}"
        warnings.warn(message)
        skipped_items.append(message)
    except Exception as exc:
        message = f"dopri5 failed for rtol={rtol}: {type(exc).__name__}: {exc}"
        warnings.warn(message)
        skipped_items.append(message)

solver_table = pd.DataFrame(solver_rows)
save_csv(solver_table, "table03_01_solver_diagnostics.csv")
fig, ax = plot_nfe_vs_error(solver_table, y="mmd_20d")
ax.set_ylabel("MMD to target in 20D PC space")
ax.set_title("NFE versus endpoint error")
save_figure(fig, "fig03_10_nfe_vs_endpoint_error.png")
solver_table

In [ ]:
display_saved_figure(FIG_DIR / "fig03_10_nfe_vs_endpoint_error.png", width=650)

In [ ]:
display_table(pd.read_csv(TABLE_DIR / "table03_01_solver_diagnostics.csv"), n=10)

## 9. CNF-Endpoint Baseline: Same Endpoint Objective, ODE-in-the-Loop Training

This baseline uses the same EB 20D train/validation split and the same endpoint-pair source as CFM, but changes the training control flow.

- CFM trains by local velocity regression at sampled interpolation times and does not solve the learned ODE inside the loss.
- CNF-Endpoint predicts `x1_pred = Phi_theta^{0->1}(x0)` and minimizes endpoint MSE against the sampled target endpoint.
- CNF-Endpoint uses `torchdiffeq.odeint_adjoint` for backpropagation through the ODE solve.
- This is not FFJORD or likelihood training. CNF maximum-likelihood with divergence/Hutchinson trace estimation is a different objective and is intentionally left out of the main comparison.

All model states, endpoint losses, MMD, and Sliced W2 values in this section are computed in 20D PC space. PHATE appears only in the sample display figure.

The CNF endpoint baseline optimizes through an ODE solve. This tests training cost and endpoint behavior, not likelihood modeling.

In [ ]:
cnf_endpoint_completed = False
cnf_endpoint_skipped_reason = None

def record_skip(item, reason):
    skipped_items.append({"item": str(item), "reason": str(reason)})


def make_seeded_pair_batch_fn(X0, X1, seed=42):
    local_rng = np.random.default_rng(seed)

    def _pair_batch_fn(batch_size):
        idx0 = local_rng.integers(0, len(X0), size=int(batch_size))
        idx1 = local_rng.integers(0, len(X1), size=int(batch_size))
        return {"x0": X0[idx0].astype(np.float32), "x1": X1[idx1].astype(np.float32)}

    return _pair_batch_fn


def endpoint_mmd_sliced_20d(samples_20d, target_20d, seed=42, n_projections=None):
    if n_projections is None:
        n_projections = 48 if SMOKE_MODE else 128
    return {
        "endpoint_mmd_20d": float(mmd_rbf(samples_20d, target_20d)),
        "sliced_w2_20d": float(sliced_wasserstein_distance(samples_20d, target_20d, n_projections=n_projections, seed=seed)),
    }


def eval_cfm_endpoint_20d(model, x0_np, target_np, n_steps=30, seed=42):
    model.eval()
    with torch.no_grad():
        x0_t = torch.as_tensor(x0_np, dtype=torch.float32, device=DEVICE)
        endpoint_t, _, nfe = euler_sample(model, x0_t, n_steps=int(n_steps), return_traj=False)
    endpoint_np = as_np(endpoint_t)
    metrics = endpoint_mmd_sliced_20d(endpoint_np, target_np, seed=seed)
    return endpoint_np, int(nfe), metrics


def eval_ode_endpoint_20d(model, x0_np, target_np, rtol=1e-3, seed=42):
    model.eval()
    with torch.no_grad():
        x0_t = torch.as_tensor(x0_np, dtype=torch.float32, device=DEVICE)
        endpoint_t, _, nfe = odeint_sample(model, x0_t, rtol=float(rtol), atol=float(rtol), method="dopri5")
    endpoint_np = as_np(endpoint_t)
    metrics = endpoint_mmd_sliced_20d(endpoint_np, target_np, seed=seed)
    return endpoint_np, int(nfe), metrics


class EndpointODEFunc(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.nfe = 0

    def reset_nfe(self):
        self.nfe = 0

    def forward(self, t, x):
        self.nfe += 1
        tt = torch.ones((x.shape[0], 1), device=x.device, dtype=x.dtype) * t
        return self.model(x, tt)


cnf_steps = 80 if QUICK_MODE else 600
cnf_batch_size = 64 if QUICK_MODE else 128
cnf_rtol = 1e-3 if QUICK_MODE else 1e-4
cnf_log_every = 20 if QUICK_MODE else 50
cfm_e1_sample_steps = 30 if QUICK_MODE else 50

In [ ]:
if SMOKE_MODE:
    cnf_steps = 6
    cnf_batch_size = 32
    cnf_rtol = 2e-3
    cnf_log_every = 3
    cfm_e1_sample_steps = 8

cnf_eval_n = min(360 if QUICK_MODE else 900, len(X0_val), len(X1_val))
if SMOKE_MODE:
    cnf_eval_n = min(90, len(X0_val), len(X1_val))
eval_idx0 = subsample_idx(len(X0_val), cnf_eval_n, seed=SEED + 200)
eval_target_idx = subsample_idx(len(X1_val), cnf_eval_n, seed=SEED + 201)
e1_eval_x0_20d = X0_val[eval_idx0].astype(np.float32)
e1_eval_target_20d = X1_val[eval_target_idx].astype(np.float32)

# Fair lightweight CFM probe for the training-cost curve. It uses the same architecture and pair source as CNF-Endpoint.
set_seed(SEED + 210)
cfm_e1_model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
cfm_e1_optimizer = torch.optim.Adam(cfm_e1_model.parameters(), lr=1e-3)
cfm_e1_pair_batch = make_seeded_pair_batch_fn(X0_train, X1_train, seed=SEED + 211)
cfm_e1_rows = []
start = time.perf_counter()
last_time = start
last_step = 0
for step in range(1, cnf_steps + 1):
    batch = cfm_e1_pair_batch(cnf_batch_size)
    x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
    x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
    loss = cfm_loss_from_pairs(cfm_e1_model, x0, x1)
    cfm_e1_optimizer.zero_grad()
    loss.backward()
    cfm_e1_optimizer.step()
    if step == 1 or step % cnf_log_every == 0 or step == cnf_steps:
        now = time.perf_counter()
        _, sample_nfe, met = eval_cfm_endpoint_20d(cfm_e1_model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=cfm_e1_sample_steps, seed=SEED + step)
        cfm_e1_rows.append({
            "method": "CFM",
            "step": int(step),
            "wall_time_sec": float(now - start),
            "sec_per_step": float((now - last_time) / max(1, step - last_step)),
            "final_train_loss_20d": float(loss.detach().cpu()),
            "val_endpoint_mmd_20d": met["endpoint_mmd_20d"],
            "val_sliced_w2_20d": met["sliced_w2_20d"],
            "train_nfe_per_step": 1,
            "sample_nfe": int(sample_nfe),
        })
        last_time = now
        last_step = step

cnf_rows = []
cnf_model = None

In [ ]:
try:
    from torchdiffeq import odeint_adjoint

    set_seed(SEED + 220)
    cnf_model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
    cnf_optimizer = torch.optim.Adam(cnf_model.parameters(), lr=1e-3)
    cnf_pair_batch = make_seeded_pair_batch_fn(X0_train, X1_train, seed=SEED + 211)
    ode_func = EndpointODEFunc(cnf_model)
    t_grid_train = torch.tensor([0.0, 1.0], dtype=torch.float32, device=DEVICE)
    start = time.perf_counter()
    last_time = start
    last_step = 0
    for step in range(1, cnf_steps + 1):
        batch = cnf_pair_batch(cnf_batch_size)
        x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
        x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
        ode_func.reset_nfe()
        pred_traj = odeint_adjoint(
            ode_func,
            x0,
            t_grid_train,
            rtol=float(cnf_rtol),
            atol=float(cnf_rtol),
            method="dopri5",
            adjoint_params=tuple(cnf_model.parameters()),
        )
        x1_pred = pred_traj[-1]
        loss = ((x1_pred - x1) ** 2).mean(dim=-1).mean()
        cnf_optimizer.zero_grad()
        loss.backward()
        cnf_optimizer.step()
        train_nfe = int(ode_func.nfe)
        if step == 1 or step % cnf_log_every == 0 or step == cnf_steps:
            now = time.perf_counter()
            _, sample_nfe, met = eval_ode_endpoint_20d(cnf_model, e1_eval_x0_20d, e1_eval_target_20d, rtol=cnf_rtol, seed=SEED + 500 + step)
            cnf_rows.append({
                "method": "CNF-Endpoint",
                "step": int(step),
                "wall_time_sec": float(now - start),
                "sec_per_step": float((now - last_time) / max(1, step - last_step)),
                "final_train_loss_20d": float(loss.detach().cpu()),
                "val_endpoint_mmd_20d": met["endpoint_mmd_20d"],
                "val_sliced_w2_20d": met["sliced_w2_20d"],
                "train_nfe_per_step": int(train_nfe),
                "sample_nfe": int(sample_nfe),
            })
            last_time = now
            last_step = step
    cnf_endpoint_completed = True
except Exception as exc:
    cnf_endpoint_skipped_reason = f"{type(exc).__name__}: {exc}"
    record_skip("CNF-Endpoint baseline", cnf_endpoint_skipped_reason)

cfm_e1_history = pd.DataFrame(cfm_e1_rows)
cnf_e1_history = pd.DataFrame(cnf_rows)
e1_history = pd.concat([cfm_e1_history, cnf_e1_history], ignore_index=True)

# Final sample metrics and display endpoints for the E1 table/figure.
cfm_e1_endpoint_20d, cfm_e1_sample_nfe, cfm_e1_metrics = eval_cfm_endpoint_20d(
    cfm_e1_model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=cfm_e1_sample_steps, seed=SEED + 230
)

In [ ]:
summary_rows = [{
    "method": "CFM",
    "train_objective": "local_velocity_regression_20D",
    "train_uses_ode": False,
    "steps": int(cnf_steps),
    "batch_size": int(cnf_batch_size),
    "wall_time_sec": float(cfm_e1_history["wall_time_sec"].iloc[-1]),
    "mean_sec_per_step": float(cfm_e1_history["wall_time_sec"].iloc[-1] / max(cnf_steps, 1)),
    "train_nfe_per_step": 1,
    "final_train_loss_20d": float(cfm_e1_history["final_train_loss_20d"].iloc[-1]),
    "endpoint_mmd_20d": cfm_e1_metrics["endpoint_mmd_20d"],
    "sliced_w2_20d": cfm_e1_metrics["sliced_w2_20d"],
    "sample_nfe": int(cfm_e1_sample_nfe),
}]

if cnf_endpoint_completed and cnf_model is not None:
    cnf_e1_endpoint_20d, cnf_e1_sample_nfe, cnf_e1_metrics = eval_ode_endpoint_20d(
        cnf_model, e1_eval_x0_20d, e1_eval_target_20d, rtol=cnf_rtol, seed=SEED + 231
    )
    summary_rows.append({
        "method": "CNF-Endpoint",
        "train_objective": "endpoint_mse_after_adjoint_ode_solve_20D",
        "train_uses_ode": True,
        "steps": int(cnf_steps),
        "batch_size": int(cnf_batch_size),
        "wall_time_sec": float(cnf_e1_history["wall_time_sec"].iloc[-1]),
        "mean_sec_per_step": float(cnf_e1_history["wall_time_sec"].iloc[-1] / max(cnf_steps, 1)),
        "train_nfe_per_step": float(cnf_e1_history["train_nfe_per_step"].mean()),
        "final_train_loss_20d": float(cnf_e1_history["final_train_loss_20d"].iloc[-1]),
        "endpoint_mmd_20d": cnf_e1_metrics["endpoint_mmd_20d"],
        "sliced_w2_20d": cnf_e1_metrics["sliced_w2_20d"],
        "sample_nfe": int(cnf_e1_sample_nfe),
    })
else:
    cnf_e1_endpoint_20d = None
    cnf_e1_sample_nfe = None

E1_table = pd.DataFrame(summary_rows)
save_csv(E1_table, "tableE1_cfm_vs_cnf_endpoint.csv")

fig, ax = plt.subplots(figsize=(5.6, 3.5))
for method, group in e1_history.groupby("method", sort=False):
    ax.plot(group["wall_time_sec"], group["val_endpoint_mmd_20d"], marker="o", linewidth=1.5, markersize=3.0, label=method)
ax.set_xlabel("training wall-clock seconds")
ax.set_ylabel("validation endpoint MMD in 20D")
ax.set_title("CFM local regression versus CNF-Endpoint adjoint training")
ax.grid(axis="y", color="0.92", linewidth=0.7)
ax.legend(frameon=False)
save_figure(fig, "figE1_cfm_vs_cnf_endpoint_training_cost.png")

if cnf_e1_endpoint_20d is not None:
    cfm_plot = phate_projector.predict(cfm_e1_endpoint_20d)
    cnf_plot = phate_projector.predict(cnf_e1_endpoint_20d)
    target_plot = phate_projector.predict(e1_eval_target_20d)
    xlim, ylim = robust_limits(target_plot, cfm_plot, cnf_plot, margin=0.10)
    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.8), sharex=True, sharey=True)
    panels = [(axes[0], cfm_plot, "A. CFM endpoints\nPHATE display only", "#2F6B5E"), (axes[1], cnf_plot, "B. CNF-Endpoint endpoints\nPHATE display only", "#B9352B")]
    for ax, pts, title, color in panels:
        ax.scatter(target_plot[:, 0], target_plot[:, 1], s=7, color="0.65", alpha=0.24, linewidths=0, label="held-out target")
        ax.scatter(pts[:, 0], pts[:, 1], s=8, color=color, alpha=0.56, linewidths=0, label="generated")
        format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title=title)
        ax.legend(frameon=False, loc="best")
        if ax is not axes[0]:
            ax.set_ylabel("")
    fig.suptitle("Endpoint samples from 20D models projected for PHATE visualization")
    save_figure(fig, "figE1_cfm_vs_cnf_endpoint_samples_phate.png")
else:
    record_skip("figE1_cfm_vs_cnf_endpoint_samples_phate.png", "CNF-Endpoint did not complete")

E1_table

In [ ]:
display_saved_figures([
    FIG_DIR / "figE1_cfm_vs_cnf_endpoint_samples_phate.png",
    FIG_DIR / "figE1_cfm_vs_cnf_endpoint_training_cost.png",
], width=650)

In [ ]:
display_table(pd.read_csv(TABLE_DIR / "tableE1_cfm_vs_cnf_endpoint.csv"), n=8)

## 10. Time Sampling Strategy Ablation

This ablation changes only the training-time interpolation-time distribution. All models are trained on the same EB 20D train/validation split with the same architecture and endpoint-pair source.

The empirical distribution plot is sampled directly from the time samplers used by the training loop. Metrics and validation MSE are in 20D PC space.

Time sampling changes the distribution of interpolation times used by the same local velocity objective.

In [ ]:
time_sampling_completed = False

time_strategy_specs = [
    ("uniform", {}),
    ("logit_normal_sigma_0.5", {"sigma": 0.5}),
    ("logit_normal_sigma_1.0", {"sigma": 1.0}),
    ("logit_normal_sigma_2.0", {"sigma": 2.0}),
    ("beta_2_2", {"alpha": 2.0, "beta": 2.0}),
    ("beta_0.5_0.5", {"alpha": 0.5, "beta": 0.5}),
    ("cosine", {}),
]


def sample_t_numpy(strategy, n, seed=42):
    local_rng = np.random.default_rng(seed)
    if strategy == "uniform":
        t = local_rng.random(n)
    elif strategy.startswith("logit_normal"):
        sigma = dict(time_strategy_specs)[strategy]["sigma"]
        z = local_rng.normal(loc=0.0, scale=float(sigma), size=n)
        t = 1.0 / (1.0 + np.exp(-z))
    elif strategy == "beta_2_2":
        t = local_rng.beta(2.0, 2.0, size=n)
    elif strategy == "beta_0.5_0.5":
        t = local_rng.beta(0.5, 0.5, size=n)
    elif strategy == "cosine":
        u = local_rng.random(n)
        t = 0.5 * (1.0 - np.cos(np.pi * u))
    else:
        raise ValueError(strategy)
    return np.clip(t.astype(np.float32), 1e-4, 1.0 - 1e-4)


def sample_t_torch(strategy, batch_size, device, seed=None):
    # Numpy-backed sampling keeps every strategy exactly aligned with the empirical-density plot.
    if seed is None:
        seed = np.random.randint(0, 2**31 - 1)
    arr = sample_t_numpy(strategy, int(batch_size), seed=seed)
    return torch.as_tensor(arr[:, None], dtype=torch.float32, device=device)

In [ ]:
def train_cfm_with_time_strategy(strategy, steps, batch_size, log_every, seed_offset=0):
    set_seed(SEED + 300 + seed_offset)
    model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    local_pair_batch = make_seeded_pair_batch_fn(X0_train, X1_train, seed=SEED + 310 + seed_offset)
    rows = []
    start = time.perf_counter()
    last_time = start
    last_step = 0
    for step in range(1, int(steps) + 1):
        batch = local_pair_batch(batch_size)
        x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
        x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
        t_batch = sample_t_torch(strategy, batch_size, DEVICE, seed=SEED + 320 + seed_offset * 100000 + step)
        loss = cfm_loss_from_pairs(model, x0, x1, s=t_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step == 1 or step % int(log_every) == 0 or step == int(steps):
            now = time.perf_counter()
            val_mse = val_cfm_mse(model, val_x0, val_x1, val_t_grid, device=DEVICE, max_eval_pairs=900 if not SMOKE_MODE else 120)
            rows.append({
                "strategy": strategy,
                "step": int(step),
                "train_loss_20d": float(loss.detach().cpu()),
                "val_mse_20d": float(val_mse),
                "wall_time_sec": float(now - start),
                "sec_per_step": float((now - last_time) / max(1, step - last_step)),
            })
            last_time = now
            last_step = step
    return model, pd.DataFrame(rows)

# Empirical t distributions.
fig, ax = plt.subplots(figsize=(6.5, 3.7))
bins = np.linspace(0.0, 1.0, 55)
for i, (strategy, _) in enumerate(time_strategy_specs):
    samples = sample_t_numpy(strategy, 8000 if not SMOKE_MODE else 1200, seed=SEED + 330 + i)
    hist, edges = np.histogram(samples, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax.plot(centers, hist, linewidth=1.4, label=strategy)
ax.set_xlabel("sampled interpolation time t")
ax.set_ylabel("empirical density")
ax.set_title("Training-time sampling strategies for CFM")
ax.legend(frameon=False, fontsize=6, ncol=2)
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE3_time_sampling_distributions.png")

time_steps = 350 if QUICK_MODE else 1200
time_batch_size = 192 if QUICK_MODE else 256
time_log_every = 70 if QUICK_MODE else 150
time_sample_steps = 30 if QUICK_MODE else 50
if SMOKE_MODE:
    time_steps = 8
    time_batch_size = 64
    time_log_every = 4
    time_sample_steps = 8

time_histories = []
time_summary_rows = []

In [ ]:
for si, (strategy, _) in enumerate(time_strategy_specs):
    model, hist = train_cfm_with_time_strategy(strategy, time_steps, time_batch_size, time_log_every, seed_offset=si)
    endpoint_20d, sample_nfe, met = eval_cfm_endpoint_20d(model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=time_sample_steps, seed=SEED + 400 + si)
    hist["endpoint_mmd_20d"] = np.nan
    hist.loc[hist.index[-1], "endpoint_mmd_20d"] = met["endpoint_mmd_20d"]
    time_histories.append(hist)
    time_summary_rows.append({
        "strategy": strategy,
        "steps": int(time_steps),
        "final_train_loss_20d": float(hist["train_loss_20d"].iloc[-1]),
        "final_val_mse_20d": float(hist["val_mse_20d"].iloc[-1]),
        "endpoint_mmd_20d": met["endpoint_mmd_20d"],
        "sliced_w2_20d": met["sliced_w2_20d"],
        "sample_nfe": int(sample_nfe),
        "wall_time_sec": float(hist["wall_time_sec"].iloc[-1]),
    })

time_history = pd.concat(time_histories, ignore_index=True)
time_table = pd.DataFrame(time_summary_rows)
save_csv(time_table, "tableE3_time_sampling_ablation.csv")

fig, ax = plt.subplots(figsize=(6.4, 3.8))
for strategy, group in time_history.groupby("strategy", sort=False):
    ax.plot(group["step"], group["val_mse_20d"], linewidth=1.35, label=strategy)
ax.set_xlabel("training step")
ax.set_ylabel("validation CFM MSE in 20D")
ax.set_title("Time sampling ablation: validation velocity-regression MSE")
ax.grid(axis="y", color="0.92", linewidth=0.7)
ax.legend(frameon=False, fontsize=6, ncol=2)
save_figure(fig, "figE3_time_sampling_val_mse.png")

fig, ax = plt.subplots(figsize=(6.6, 3.6))
order = time_table.sort_values("endpoint_mmd_20d")["strategy"].tolist()
sns.barplot(data=time_table, x="strategy", y="endpoint_mmd_20d", order=order, ax=ax, color="#2F6B5E")
ax.set_xlabel("time sampling strategy")
ax.set_ylabel("endpoint MMD in 20D")
ax.set_title("Final endpoint MMD after short CFM runs")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE3_time_sampling_endpoint_mmd.png")

fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6))
sns.barplot(data=time_table, x="strategy", y="endpoint_mmd_20d", order=order, ax=axes[0], color="#2F6B5E")
sns.barplot(data=time_table, x="strategy", y="sliced_w2_20d", order=order, ax=axes[1], color="#B9352B")
for ax, ylabel in zip(axes, ["endpoint MMD in 20D", "Sliced W2 in 20D"]):
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=40)
    ax.grid(axis="y", color="0.92", linewidth=0.7)
axes[0].set_title("MMD")
axes[1].set_title("Sliced W2")
fig.suptitle("Time sampling ablation final endpoint metrics")
save_figure(fig, "figE3_time_sampling_final_bar.png")
time_sampling_completed = True
time_table

In [ ]:
display_saved_figures([
    FIG_DIR / "figE3_time_sampling_distributions.png",
    FIG_DIR / "figE3_time_sampling_val_mse.png",
    FIG_DIR / "figE3_time_sampling_endpoint_mmd.png",
    FIG_DIR / "figE3_time_sampling_final_bar.png",
], width=650)

In [ ]:
display_table(pd.read_csv(TABLE_DIR / "tableE3_time_sampling_ablation.csv"), n=8)

## 11. Network Capacity Ablation

This ablation changes only the `VelocityMLP` capacity. Input and output dimensions remain 20, and every model uses the same EB 20D train/validation split and random endpoint-pair source.

Metrics are final validation CFM MSE, endpoint MMD in 20D, Sliced W2 in 20D, parameter count, and wall-clock time.

Capacity ablation keeps the EB task fixed and varies model size.

In [ ]:
capacity_ablation_completed = False

capacity_configs = [
    {"config": "tiny", "hidden_dim": 64, "hidden_layers": 3},
    {"config": "small", "hidden_dim": 128, "hidden_layers": 4},
    {"config": "medium", "hidden_dim": 256, "hidden_layers": 4},
]
if not QUICK_MODE:
    capacity_configs.append({"config": "large", "hidden_dim": 512, "hidden_layers": 6})

capacity_steps = 300 if QUICK_MODE else 1200
capacity_batch_size = 192 if QUICK_MODE else 256
capacity_log_every = 100 if QUICK_MODE else 200
capacity_sample_steps = 30 if QUICK_MODE else 50
if SMOKE_MODE:
    capacity_steps = 8
    capacity_batch_size = 64
    capacity_log_every = 4
    capacity_sample_steps = 8

capacity_rows = []
capacity_histories = []
for ci, cfg in enumerate(capacity_configs):
    set_seed(SEED + 500 + ci)
    model = VelocityMLP(x_dim=20, hidden_dim=cfg["hidden_dim"], hidden_layers=cfg["hidden_layers"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    local_pair_batch = make_seeded_pair_batch_fn(X0_train, X1_train, seed=SEED + 510 + ci)
    start = time.perf_counter()
    last_loss = np.nan
    hist_rows = []
    for step in range(1, int(capacity_steps) + 1):
        batch = local_pair_batch(capacity_batch_size)
        x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
        x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
        loss = cfm_loss_from_pairs(model, x0, x1)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
        if step == 1 or step % capacity_log_every == 0 or step == int(capacity_steps):
            val_mse = val_cfm_mse(model, val_x0, val_x1, val_t_grid, device=DEVICE, max_eval_pairs=900 if not SMOKE_MODE else 120)
            hist_rows.append({"config": cfg["config"], "step": int(step), "train_loss_20d": last_loss, "val_mse_20d": val_mse})
    wall = time.perf_counter() - start
    endpoint_20d, sample_nfe, met = eval_cfm_endpoint_20d(model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=capacity_sample_steps, seed=SEED + 540 + ci)
    hist = pd.DataFrame(hist_rows)
    capacity_histories.append(hist)
    capacity_rows.append({
        "config": cfg["config"],
        "input_dim": 20,
        "output_dim": 20,
        "hidden_dim": int(cfg["hidden_dim"]),
        "hidden_layers": int(cfg["hidden_layers"]),
        "n_parameters": int(count_parameters(model)),
        "lr": 1e-3,
        "batch_size": int(capacity_batch_size),
        "steps": int(capacity_steps),
        "final_train_mse_20d": float(last_loss),
        "final_val_mse_20d": float(hist["val_mse_20d"].iloc[-1]),
        "endpoint_mmd_20d": met["endpoint_mmd_20d"],
        "sliced_w2_20d": met["sliced_w2_20d"],
        "sample_nfe": int(sample_nfe),
        "wall_time_sec": float(wall),
    })

capacity_table = pd.DataFrame(capacity_rows)
capacity_history = pd.concat(capacity_histories, ignore_index=True)
save_csv(capacity_table, "tableT2_training_hyperparams_capacity.csv")

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(capacity_table["n_parameters"], capacity_table["final_val_mse_20d"], marker="o", linewidth=1.5, color="#2F6B5E")

In [ ]:
for _, row in capacity_table.iterrows():
    ax.text(row["n_parameters"], row["final_val_mse_20d"], row["config"], fontsize=7, ha="left", va="bottom")
ax.set_xscale("log")
ax.set_xlabel("trainable parameters")
ax.set_ylabel("final validation MSE in 20D")
ax.set_title("Capacity ablation: validation velocity-regression error")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE2_capacity_val_mse.png")

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(capacity_table["n_parameters"], capacity_table["endpoint_mmd_20d"], marker="s", linewidth=1.5, color="#B9352B")
for _, row in capacity_table.iterrows():
    ax.text(row["n_parameters"], row["endpoint_mmd_20d"], row["config"], fontsize=7, ha="left", va="bottom")
ax.set_xscale("log")
ax.set_xlabel("trainable parameters")
ax.set_ylabel("endpoint MMD in 20D")
ax.set_title("Capacity ablation: endpoint distribution error")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE2_capacity_endpoint_mmd.png")
capacity_ablation_completed = True
capacity_table

In [ ]:
display_saved_figures([
    FIG_DIR / "figE2_capacity_val_mse.png",
    FIG_DIR / "figE2_capacity_endpoint_mmd.png",
], width=650)

In [ ]:
display_table(pd.read_csv(TABLE_DIR / "tableT2_training_hyperparams_capacity.csv"), n=8)

## 12. Trajectory Straightness in 20D

This diagnostic uses the main EB CFM model and measures trajectory straightness in 20D PC space. The ratio is

`sum_k ||x_{k+1}-x_k|| / ||x_K-x_0||`.

A value near 1 indicates a nearly straight numerical path in 20D. PHATE is used only to display representative trajectories after the 20D straightness scores are computed.

Straightness is a trajectory diagnostic in 20D PC space, not a lineage certificate.

In [ ]:
straightness_completed = False


def per_trajectory_straightness(traj, eps=1e-8):
    traj = np.asarray(traj, dtype=float)
    if traj.ndim != 3:
        raise ValueError("traj must have shape (T, N, D)")
    steps = np.linalg.norm(np.diff(traj, axis=0), axis=-1)
    path_len = steps.sum(axis=0)
    endpoint = np.linalg.norm(traj[-1] - traj[0], axis=-1)
    ratio = path_len / np.maximum(endpoint, float(eps))
    return path_len, endpoint, ratio

straight_sample_n = min(900 if QUICK_MODE else 1600, len(X0_val))
straight_steps = 50 if QUICK_MODE else 100
if SMOKE_MODE:
    straight_sample_n = min(120, len(X0_val))
    straight_steps = 10
straight_idx = subsample_idx(len(X0_val), straight_sample_n, seed=SEED + 600)
straight_x0_t = torch.as_tensor(X0_val[straight_idx], dtype=torch.float32, device=DEVICE)
eb_model.eval()
with torch.no_grad():
    straight_endpoint_t, straight_traj_t, straight_nfe = euler_sample(eb_model, straight_x0_t, n_steps=straight_steps, return_traj=True)
straight_traj_20d = as_np(straight_traj_t)
path_length_20d, endpoint_distance_20d, straight_ratio = per_trajectory_straightness(straight_traj_20d)
q25, q50, q75, q90 = np.quantile(straight_ratio, [0.25, 0.50, 0.75, 0.90])
quantile_group = np.full(straight_ratio.shape, "middle", dtype=object)
quantile_group[straight_ratio <= q25] = "lower_quantile"
quantile_group[straight_ratio >= q75] = "upper_quantile"
straight_table = pd.DataFrame({
    "cell_index": straight_idx.astype(int),
    "endpoint_distance_20d": endpoint_distance_20d,
    "path_length_20d": path_length_20d,
    "straightness_ratio_20d": straight_ratio,
    "quantile_group": quantile_group,
})
save_csv(straight_table, "tableE5_trajectory_straightness.csv")

fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.hist(straight_ratio, bins=36, color="#2F6B5E", alpha=0.78, edgecolor="white", linewidth=0.4)
for value, label, color in [(straight_ratio.mean(), "mean", "#2F6B5E"), (q50, "median", "0.20"), (q90, "90th pct", "#B9352B")]:
    ax.axvline(value, color=color, linestyle="--", linewidth=1.2)
    ax.text(value, ax.get_ylim()[1] * 0.92, label, rotation=90, va="top", ha="right", fontsize=7, color=color)
ax.set_xlabel("20D straightness ratio")
ax.set_ylabel("trajectory count")
ax.set_title("Trajectory straightness under main EB CFM sampler")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE5_straightness_hist.png")

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.scatter(endpoint_distance_20d, straight_ratio, s=9, color="#2F6B5E", alpha=0.38, linewidths=0)
ax.set_xlabel("endpoint distance in 20D")
ax.set_ylabel("straightness ratio")
ax.set_title("Endpoint displacement versus 20D path straightness")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE5_endpoint_distance_vs_straightness.png")

low_candidates = np.flatnonzero(straight_ratio <= q25)
high_candidates = np.flatnonzero(straight_ratio >= q75)
low_show = low_candidates[subsample_idx(len(low_candidates), min(8, len(low_candidates)), seed=SEED + 610)]
high_show = high_candidates[subsample_idx(len(high_candidates), min(8, len(high_candidates)), seed=SEED + 611)]
fig, ax = plt.subplots(figsize=(5.8, 4.6))
tidx = subsample_idx(len(X_target_phate), 700, seed=SEED + 612)
ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=5, color="0.70", alpha=0.18, linewidths=0, label="real target")
traj_phate_arrays = []
for j in low_show:
    pts = phate_projector.predict(straight_traj_20d[:, j, :])
    traj_phate_arrays.append(pts)
    ax.plot(pts[:, 0], pts[:, 1], color="#2F6B5E", alpha=0.58, linewidth=1.0)
    ax.scatter(pts[[0, -1], 0], pts[[0, -1], 1], s=14, color="#2F6B5E", alpha=0.70, linewidths=0)

In [ ]:
for j in high_show:
    pts = phate_projector.predict(straight_traj_20d[:, j, :])
    traj_phate_arrays.append(pts)
    ax.plot(pts[:, 0], pts[:, 1], color="#B9352B", alpha=0.60, linewidth=1.0)
    ax.scatter(pts[[0, -1], 0], pts[[0, -1], 1], s=14, color="#B9352B", alpha=0.72, linewidths=0)
all_traj_plot = np.vstack(traj_phate_arrays) if traj_phate_arrays else X_target_phate[tidx]
xlim, ylim = robust_limits(X_target_phate[tidx], all_traj_plot, margin=0.10)
format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title="Representative 20D trajectories projected to PHATE display only")
ax.plot([], [], color="#2F6B5E", label="lower straightness quantile")
ax.plot([], [], color="#B9352B", label="upper straightness quantile")
ax.legend(frameon=False, loc="best")
save_figure(fig, "figE5_representative_trajectories_phate.png")
straightness_completed = True
straight_table.describe(include="all")

In [ ]:
display_saved_figures([
    FIG_DIR / "figE5_straightness_hist.png",
    FIG_DIR / "figE5_endpoint_distance_vs_straightness.png",
    FIG_DIR / "figE5_representative_trajectories_phate.png",
], width=650)

In [ ]:
display_table(pd.read_csv(TABLE_DIR / "tableE5_trajectory_straightness.csv"), n=8)

## Paper-ready tables

The paper table exports belong here because they summarize solver, baseline, ablation, and straightness diagnostics rather than the toy or EB main training tutorial.

In [ ]:
def _round_float(x, decimals):
    return pd.to_numeric(x, errors="coerce").round(decimals)

solver_paper = solver_table.copy()
solver_paper["Solver"] = solver_paper["sampler"].astype(str)
solver_paper["Steps"] = solver_paper["steps"].astype(str)
solver_paper["NFE"] = solver_paper["nfe"].astype(int)
solver_paper["Time (ms)"] = _round_float(1000.0 * solver_paper["wall_time_sec"], 1)
solver_paper["MMD (20D) ↓"] = _round_float(solver_paper["mmd_20d"], 4)
solver_paper["Sliced W2 (20D) ↓"] = _round_float(solver_paper["sliced_w2_20d"], 3)
solver_paper["Straightness (20D)"] = _round_float(solver_paper["trajectory_straightness_20d"], 3).astype(object)
solver_paper.loc[solver_paper["sampler"].astype(str).eq("dopri5"), "Straightness (20D)"] = "N/A"
solver_paper = solver_paper[["Solver", "Steps", "NFE", "Time (ms)", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "Straightness (20D)"]]
save_paper_table(solver_paper, "paper_table03_01_solver_diagnostics")

E1_paper = E1_table.copy()
E1_paper["Method"] = E1_paper["method"]
E1_paper["Objective"] = E1_paper["train_objective"].str.replace("_", " ")
E1_paper["Uses ODE"] = E1_paper["train_uses_ode"].map({True: "yes", False: "no"})
E1_paper["Steps"] = E1_paper["steps"].astype(int)
E1_paper["Batch"] = E1_paper["batch_size"].astype(int)
E1_paper["Time (s)"] = _round_float(E1_paper["wall_time_sec"], 2)
E1_paper["NFE/step"] = _round_float(E1_paper["train_nfe_per_step"], 1)
E1_paper["MMD (20D) ↓"] = _round_float(E1_paper["endpoint_mmd_20d"], 4)
E1_paper["Sliced W2 (20D) ↓"] = _round_float(E1_paper["sliced_w2_20d"], 3)
E1_paper["Mode"] = "QUICK diagnostic" if QUICK_MODE else "full run"
E1_paper = E1_paper[["Method", "Objective", "Uses ODE", "Steps", "Batch", "Time (s)", "NFE/step", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "Mode"]]
save_paper_table(E1_paper, "paper_tableE1_cfm_vs_cnf_endpoint")

time_paper = time_table.copy()
time_paper["Strategy"] = time_paper["strategy"].map(short_strategy_label)
time_paper["Steps"] = time_paper["steps"].astype(int)
time_paper["Train MSE (20D)"] = _round_float(time_paper["final_train_loss_20d"], 3)
time_paper["Val MSE (20D)"] = _round_float(time_paper["final_val_mse_20d"], 3)
time_paper["MMD (20D) ↓"] = _round_float(time_paper["endpoint_mmd_20d"], 4)
time_paper["Sliced W2 (20D) ↓"] = _round_float(time_paper["sliced_w2_20d"], 3)
time_paper["NFE"] = time_paper["sample_nfe"].astype(int)
time_paper["Time (s)"] = _round_float(time_paper["wall_time_sec"], 2)
time_paper = time_paper[["Strategy", "Steps", "Train MSE (20D)", "Val MSE (20D)", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "NFE", "Time (s)"]]
save_paper_table(time_paper, "paper_tableE3_time_sampling_ablation")

cap_paper = capacity_table.copy()
cap_paper["Config"] = cap_paper["config"]
cap_paper["Input dim"] = cap_paper["input_dim"].astype(int)
cap_paper["Output dim"] = cap_paper["output_dim"].astype(int)
cap_paper["Hidden"] = cap_paper["hidden_dim"].astype(int)
cap_paper["Layers"] = cap_paper["hidden_layers"].astype(int)
cap_paper["Params"] = cap_paper["n_parameters"].astype(int)
cap_paper["Steps"] = cap_paper["steps"].astype(int)
cap_paper["Train MSE (20D)"] = _round_float(cap_paper["final_train_mse_20d"], 3)
cap_paper["Val MSE (20D)"] = _round_float(cap_paper["final_val_mse_20d"], 3)
cap_paper["MMD (20D) ↓"] = _round_float(cap_paper["endpoint_mmd_20d"], 4)
cap_paper["Sliced W2 (20D) ↓"] = _round_float(cap_paper["sliced_w2_20d"], 3)
cap_paper["Time (s)"] = _round_float(cap_paper["wall_time_sec"], 2)
cap_paper = cap_paper[["Config", "Input dim", "Output dim", "Hidden", "Layers", "Params", "Steps", "Train MSE (20D)", "Val MSE (20D)", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "Time (s)"]]
save_paper_table(cap_paper, "paper_tableT2_training_hyperparams_capacity")

straight_summary = pd.DataFrame([
    {
        "Metric": "Straightness ratio (20D)",
        "count": int(straight_table["straightness_ratio_20d"].count()),
        "mean": round(float(straight_table["straightness_ratio_20d"].mean()), 3),
        "median": round(float(straight_table["straightness_ratio_20d"].median()), 3),
        "std": round(float(straight_table["straightness_ratio_20d"].std()), 3),
        "q10": round(float(straight_table["straightness_ratio_20d"].quantile(0.10)), 3),
        "q90": round(float(straight_table["straightness_ratio_20d"].quantile(0.90)), 3),
        "max": round(float(straight_table["straightness_ratio_20d"].max()), 3),
    },
    {
        "Metric": "Endpoint distance (20D)",
        "count": int(straight_table["endpoint_distance_20d"].count()),
        "mean": round(float(straight_table["endpoint_distance_20d"].mean()), 3),
        "median": round(float(straight_table["endpoint_distance_20d"].median()), 3),
        "std": round(float(straight_table["endpoint_distance_20d"].std()), 3),
        "q10": round(float(straight_table["endpoint_distance_20d"].quantile(0.10)), 3),
        "q90": round(float(straight_table["endpoint_distance_20d"].quantile(0.90)), 3),
        "max": round(float(straight_table["endpoint_distance_20d"].max()), 3),
    },
])
save_paper_table(straight_summary, "paper_tableE5_trajectory_straightness_summary")

In [ ]:
paper_ready_pass_completed = True
figure_quality_notes = {
    "style": "Unified DejaVu Sans paper style, light grids only on metric plots, top/right spines removed.",
    "phate": "PHATE annotations moved to short corner notes; PHATE is display only.",
    "quick_mode": "E1/E2/E3 diagnostics are QUICK_MODE unless rerun with QUICK_MODE=False.",
    "fig03_10": "Single concise title and explicit 20D PC metric label.",
}
print({
    "paper_ready_pass_completed": paper_ready_pass_completed,
    "paper_ready_png_written": len(paper_ready_png_written),
    "paper_ready_pdf_written": len(paper_ready_pdf_written),
    "paper_tables_written": len(paper_tables_written),
})

In [ ]:
display_table(pd.read_csv(TABLE_DIR / "paper_table03_01_solver_diagnostics.csv"), n=8)

## Chapter 3 Artifact Index

The final index records every Chapter 3 artifact, including toy and main-flow artifacts owned by the previous split notebooks. The concept diagrams are tracked as manual/reference figures because they are not generated by this notebook.

In [ ]:
manual_concept_figure_paths = {
    "Fig 3.1": PROJECT_ROOT.parent / "tutorial_figure" / "ch03" / "fig03_01_training_vs_sampling_compute.png",
    "Fig 3.5": PROJECT_ROOT.parent / "tutorial_figure" / "ch03" / "fig03_05_velocity_mlp_architecture.png",
    "Fig 3.6": PROJECT_ROOT.parent / "tutorial_figure" / "ch03" / "fig03_06_minimal_cfm_training_loop.png",
    "Fig 3.11": PROJECT_ROOT.parent / "tutorial_figure" / "ch03" / "fig03_11_cfm_extension_map.png",
}
artifact_rows = [
    {"artifact_id": "Fig toy loss", "artifact_type": "figure", "path": "figures/ch03/fig_toy_loss.png", "owner": "03_1", "source_section": "toy training", "notes": "Toy-only teaching diagnostic"},
    {"artifact_id": "Fig toy evolution", "artifact_type": "figure", "path": "figures/ch03/fig_toy_evolution.png", "owner": "03_1", "source_section": "toy sampling", "notes": "Toy-only teaching diagnostic"},
    {"artifact_id": "Fig 3.2", "artifact_type": "figure", "path": "figures/ch03/fig03_02_conditional_vs_marginal_toy.png", "owner": "03_1", "source_section": "toy conditional versus marginal", "notes": "Teaching concept only"},
    {"artifact_id": "Fig 3.3", "artifact_type": "figure", "path": "figures/ch03/fig03_03_cfm_object_hierarchy_toy.png", "owner": "03_1", "source_section": "toy object hierarchy", "notes": "Teaching concept only"},
    {"artifact_id": "Fig B1", "artifact_type": "figure", "path": "figures/ch03/figB1_eb20d_train_val_loss.png", "owner": "03_2", "source_section": "EB 20D training", "notes": "Train/val CFM MSE"},
    {"artifact_id": "Fig 3.4", "artifact_type": "figure", "path": "figures/ch03/fig03_04_eb_endpoint_pairs_phate.png", "owner": "03_2", "source_section": "EB endpoint pair visualization", "notes": "PHATE display only"},
    {"artifact_id": "Fig 3.8", "artifact_type": "figure", "path": "figures/ch03/fig03_08_eb_population_evolution_phate.png", "owner": "03_2", "source_section": "EB population evolution", "notes": "20D Euler states projected with kNN"},
    {"artifact_id": "Fig 3.9", "artifact_type": "figure", "path": "figures/ch03/fig03_09_euler_step_sensitivity_phate.png", "owner": "03_2", "source_section": "Euler sensitivity", "notes": "PHATE display only; metrics in CSV"},
    {"artifact_id": "Table EB counts", "artifact_type": "table", "path": "tables/ch03/ch03_eb_timepoint_counts.csv", "owner": "03_2", "source_section": "EB data audit", "notes": "Selected timepoint counts"},
    {"artifact_id": "Table EB split", "artifact_type": "table", "path": "tables/ch03/ch03_eb20d_train_val_split.csv", "owner": "03_2", "source_section": "EB train/val split", "notes": "Split sizes"},
    {"artifact_id": "Table EB training", "artifact_type": "table", "path": "tables/ch03/ch03_eb20d_training_log.csv", "owner": "03_2", "source_section": "EB training", "notes": "Optimization history"},
    {"artifact_id": "Table Euler", "artifact_type": "table", "path": "tables/ch03/ch03_euler_step_sensitivity.csv", "owner": "03_2", "source_section": "Euler sensitivity", "notes": "NFE sensitivity metrics"},
    {"artifact_id": "Fig 3.10", "artifact_type": "figure", "path": "figures/ch03/fig03_10_nfe_vs_endpoint_error.png", "owner": "03_3", "source_section": "solver comparison", "notes": "NFE versus endpoint MMD"},
    {"artifact_id": "Table 3.1", "artifact_type": "table", "path": "tables/ch03/table03_01_solver_diagnostics.csv", "owner": "03_3", "source_section": "solver comparison", "notes": "Solver diagnostics"},
    {"artifact_id": "Fig E1 cost", "artifact_type": "figure", "path": "figures/ch03/figE1_cfm_vs_cnf_endpoint_training_cost.png", "owner": "03_3", "source_section": "CNF endpoint baseline", "notes": "CFM local regression versus adjoint endpoint training"},
    {"artifact_id": "Fig E1 samples", "artifact_type": "figure", "path": "figures/ch03/figE1_cfm_vs_cnf_endpoint_samples_phate.png", "owner": "03_3", "source_section": "CNF endpoint baseline", "notes": "PHATE display only"},
    {"artifact_id": "Table E1", "artifact_type": "table", "path": "tables/ch03/tableE1_cfm_vs_cnf_endpoint.csv", "owner": "03_3", "source_section": "CNF endpoint baseline", "notes": "Training-cost summary"},
    {"artifact_id": "Fig E2 val", "artifact_type": "figure", "path": "figures/ch03/figE2_capacity_val_mse.png", "owner": "03_3", "source_section": "capacity ablation", "notes": "Parameter count versus val MSE"},
    {"artifact_id": "Fig E2 mmd", "artifact_type": "figure", "path": "figures/ch03/figE2_capacity_endpoint_mmd.png", "owner": "03_3", "source_section": "capacity ablation", "notes": "Parameter count versus endpoint MMD"},
    {"artifact_id": "Table T2", "artifact_type": "table", "path": "tables/ch03/tableT2_training_hyperparams_capacity.csv", "owner": "03_3", "source_section": "capacity ablation", "notes": "Capacity hyperparameters and metrics"},
    {"artifact_id": "Fig E3 distributions", "artifact_type": "figure", "path": "figures/ch03/figE3_time_sampling_distributions.png", "owner": "03_3", "source_section": "time sampling", "notes": "Sampled t distributions"},
    {"artifact_id": "Fig E3 val", "artifact_type": "figure", "path": "figures/ch03/figE3_time_sampling_val_mse.png", "owner": "03_3", "source_section": "time sampling", "notes": "Validation MSE curves"},
    {"artifact_id": "Fig E3 mmd", "artifact_type": "figure", "path": "figures/ch03/figE3_time_sampling_endpoint_mmd.png", "owner": "03_3", "source_section": "time sampling", "notes": "Endpoint MMD"},
    {"artifact_id": "Fig E3 bars", "artifact_type": "figure", "path": "figures/ch03/figE3_time_sampling_final_bar.png", "owner": "03_3", "source_section": "time sampling", "notes": "Final MMD and Sliced W2"},
    {"artifact_id": "Table E3", "artifact_type": "table", "path": "tables/ch03/tableE3_time_sampling_ablation.csv", "owner": "03_3", "source_section": "time sampling", "notes": "Time sampling final metrics"},
    {"artifact_id": "Fig E5 hist", "artifact_type": "figure", "path": "figures/ch03/figE5_straightness_hist.png", "owner": "03_3", "source_section": "straightness", "notes": "20D straightness histogram"},
    {"artifact_id": "Fig E5 trajectories", "artifact_type": "figure", "path": "figures/ch03/figE5_representative_trajectories_phate.png", "owner": "03_3", "source_section": "straightness", "notes": "PHATE display only"},
    {"artifact_id": "Fig E5 scatter", "artifact_type": "figure", "path": "figures/ch03/figE5_endpoint_distance_vs_straightness.png", "owner": "03_3", "source_section": "straightness", "notes": "Endpoint distance versus straightness"},
    {"artifact_id": "Table E5", "artifact_type": "table", "path": "tables/ch03/tableE5_trajectory_straightness.csv", "owner": "03_3", "source_section": "straightness", "notes": "Per-trajectory 20D straightness"},
]
for fig_id, fig_path in manual_concept_figure_paths.items():
    artifact_rows.append({"artifact_id": fig_id, "artifact_type": "figure", "path": str(fig_path), "owner": "manual", "source_section": "manual concept diagram", "notes": "Manual/reference concept figure"})
artifact_index = pd.DataFrame(artifact_rows)
artifact_index["exists"] = artifact_index["path"].map(lambda p: Path(p).exists() if Path(p).is_absolute() else (PROJECT_ROOT / p).exists())
artifact_index["status"] = np.where(artifact_index["exists"], "written", "missing")
artifact_index.loc[artifact_index["owner"].eq("manual") & artifact_index["exists"], "status"] = "external_local"
artifact_index.loc[artifact_index["owner"].eq("manual") & ~artifact_index["exists"], "status"] = "external_missing"
main_text_ids = {"Fig 3.2", "Fig 3.3", "Fig 3.4", "Fig 3.8", "Fig 3.9", "Fig 3.10", "Fig B1", "Table 3.1"}
artifact_index["include_in_main_text"] = np.where(artifact_index["artifact_id"].isin(main_text_ids), "yes", "no")
artifact_index["include_in_appendix"] = np.where(artifact_index["artifact_id"].str.contains("E|T2", regex=True), "yes", "no")
save_csv(artifact_index, "ch03_artifact_index.csv")
artifact_index_written = bool((TABLE_DIR / "ch03_artifact_index.csv").exists())
display(artifact_index)

## Validation Notes and Summary

This summary is the split-aware replacement for the old monolithic Chapter 3 summary. It records which notebook owns each artifact and which claim boundaries remain.

In [ ]:
def normalize_skipped_items(items):
    normalized = []
    for item in items:
        if isinstance(item, dict):
            normalized.append({"item": str(item.get("item", "unknown")), "reason": str(item.get("reason", ""))})
        else:
            normalized.append({"item": str(item), "reason": "recorded by earlier section"})
    return normalized

own_code_figure_paths = [
    "figures/ch03/fig03_10_nfe_vs_endpoint_error.png",
    "figures/ch03/figE1_cfm_vs_cnf_endpoint_training_cost.png",
    "figures/ch03/figE1_cfm_vs_cnf_endpoint_samples_phate.png",
    "figures/ch03/figE3_time_sampling_distributions.png",
    "figures/ch03/figE3_time_sampling_val_mse.png",
    "figures/ch03/figE3_time_sampling_endpoint_mmd.png",
    "figures/ch03/figE3_time_sampling_final_bar.png",
    "figures/ch03/figE2_capacity_val_mse.png",
    "figures/ch03/figE2_capacity_endpoint_mmd.png",
    "figures/ch03/figE5_straightness_hist.png",
    "figures/ch03/figE5_representative_trajectories_phate.png",
    "figures/ch03/figE5_endpoint_distance_vs_straightness.png",
]
own_table_paths = [
    "tables/ch03/table03_01_solver_diagnostics.csv",
    "tables/ch03/tableE1_cfm_vs_cnf_endpoint.csv",
    "tables/ch03/tableE3_time_sampling_ablation.csv",
    "tables/ch03/tableT2_training_hyperparams_capacity.csv",
    "tables/ch03/tableE5_trajectory_straightness.csv",
    "tables/ch03/ch03_artifact_index.csv",
]
paper_table_paths = [
    "tables/ch03/paper_table03_01_solver_diagnostics.csv",
    "tables/ch03/paper_table03_01_solver_diagnostics.tex",
    "tables/ch03/paper_table03_01_solver_diagnostics.md",
    "tables/ch03/paper_tableE1_cfm_vs_cnf_endpoint.csv",
    "tables/ch03/paper_tableE1_cfm_vs_cnf_endpoint.tex",
    "tables/ch03/paper_tableE3_time_sampling_ablation.csv",
    "tables/ch03/paper_tableE3_time_sampling_ablation.tex",
    "tables/ch03/paper_tableT2_training_hyperparams_capacity.csv",
    "tables/ch03/paper_tableT2_training_hyperparams_capacity.tex",
    "tables/ch03/paper_tableE5_trajectory_straightness_summary.csv",
    "tables/ch03/paper_tableE5_trajectory_straightness_summary.tex",
]
validation_checks = {
    "eb_training_state_dim_is_20": bool(X0_train.shape[1] == 20 and X1_train.shape[1] == 20),
    "phate_state_dim_is_2_for_plots_only": bool(X0_train_phate.shape[1] == 2 and X1_train_phate.shape[1] == 2),
    "main_model_cache_exists": bool(MAIN_MODEL_PATH.exists()),
    "solver_table_exists": bool((TABLE_DIR / "table03_01_solver_diagnostics.csv").exists()),
    "artifact_index_exists": bool((TABLE_DIR / "ch03_artifact_index.csv").exists()),
    "own_code_figures_written": bool(all((PROJECT_ROOT / rel).exists() for rel in own_code_figure_paths)),
    "own_tables_written": bool(all((PROJECT_ROOT / rel).exists() for rel in own_table_paths)),
    "paper_tables_written": bool(all((PROJECT_ROOT / rel).exists() for rel in paper_table_paths)),
}
summary = {
    "quick_mode": QUICK_MODE,
    "smoke_mode": SMOKE_MODE,
    "device": DEVICE,
    "seed": SEED,
    "source_time": source_time,
    "target_time": target_time,
    "split_notebooks": [
        "03_1_toy_flow_matching_from_scratch.ipynb",
        "03_2_eb20d_main_flow_matching.ipynb",
        "03_3_eb20d_baselines_ablations_and_claim_audit.ipynb",
    ],
    "eb_training_space": "20D PC X_cost",
    "eb_visualization_space": "PHATE X_plot only",
    "phate_used_for_training_or_metrics": False,
    "toy_used_for_eb_claims": False,
    "cnf_endpoint_completed": bool(globals().get("cnf_endpoint_completed", False)),
    "time_sampling_completed": bool(globals().get("time_sampling_completed", False)),
    "capacity_ablation_completed": bool(globals().get("capacity_ablation_completed", False)),
    "straightness_completed": bool(globals().get("straightness_completed", False)),
    "artifact_index_written": bool(globals().get("artifact_index_written", False)),
    "paper_ready_pass_completed": bool(globals().get("paper_ready_pass_completed", False)),
    "known_limitations": [
        "QUICK_MODE diagnostic results need full rerun before final submission.",
        "PHATE display uses kNN projection for generated samples.",
        "CNF-Endpoint is endpoint-loss adjoint diagnostic, not FFJORD likelihood.",
    ],
    "expected_code_figures": own_code_figure_paths,
    "expected_tables": own_table_paths,
    "expected_paper_tables": paper_table_paths,
    "skipped_items": normalize_skipped_items(skipped_items),
    "validation_checks": validation_checks,
}
summary_path = OUT_DIR / "ch03_flow_matching_from_scratch_run_summary.json"
summary_path.write_text(json.dumps(ch03.json_ready(summary), indent=2), encoding="utf-8")
print(json.dumps(ch03.json_ready(summary), indent=2))

## Artifact audit

This notebook owns the solver/baseline/ablation/straightness artifacts plus the all-Chapter-3 audit files.

In [ ]:
expected_figures = [PROJECT_ROOT / rel for rel in own_code_figure_paths]
expected_tables = [PROJECT_ROOT / rel for rel in own_table_paths + paper_table_paths]
expected_outputs = [OUT_DIR / "ch03_flow_matching_from_scratch_run_summary.json"]
artifact_manifest = ch03.check_required_artifacts(expected_figures=expected_figures, expected_tables=expected_tables, expected_outputs=expected_outputs)
if not artifact_manifest["exists"].all():
    raise FileNotFoundError("Missing EB baseline/ablation/audit artifacts")
ch03.save_csv(OUT_DIR / "artifact_manifest_03_3_eb20d_baselines_ablations_and_claim_audit.csv", artifact_manifest)
display(artifact_manifest)